<br>

# ANÁLISIS DISCRIMINANTE CUADRÁTICO Y LINEAL

### QDA
Clasificador probab. model a ccad clase como distr- gaussiana con su propia matriz de covariancas, generando limites de decision cuadráticos



PARA EL EXAMEN:
    SABER QUE ES EL REG-PARAM, LA TOLERANCIA
                    reg_param -> interpola matrixz varianzas con identidad, regularizar la clase
                    tolerancia(LDA) -> regular/suavizar las diferencias entre clases (repartir)


    SABER UTILIZAR QDA / LDA EN UN EXPERIMENTO


In [1]:
import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
K = 400; pca = PCA(n_components=K); X_train = pca.fit_transform(X_train); X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# X_train.shape: 60000 x 400 parámetros    
# y_train.shape: 60000 clases
# X_test.shape: 10000 x 400 parámetros
# y_test.shape: 10000 clases

(60000, 400) (60000,) (10000, 400) (10000,)


In [2]:
# loguniform: distribución que seguiremos para muestrear valores de reg_param

from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)

array([0.00214746, 0.00011821, 0.01239803, 0.00038248, 0.00013371])

In [3]:
# Precisión del test: tras aprender reg_param y ajustar QDA con el valor aprendido

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time
start = time.time()

clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32)
acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 95.95% con 0.0599
Tiempo (hh:mm:ss): 00:02:32


In [4]:
import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
K = 400; pca = PCA(n_components=K); X_train = pca.fit_transform(X_train); X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 400) (60000,) (10000, 400) (10000,)


In [5]:
# tol: suavizado de varianzas y covarianzas

In [6]:
# Precisón en test: tras aprender tol y ajustar LDA con el valor aprendido

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform
import time

start = time.time()
clf = LinearDiscriminantAnalysis()
G = {'tol': loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 87.30% con 0.0031
Tiempo (hh:mm:ss): 00:00:17


In [7]:
# Ejercicios: 
# FASHION-MNIST (K = 600)

# Normalizamos las imágenes y apliamos PCA para reducir la dim de 784 a K componentes principales
import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("zalando-datasets/fashion_mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 600; pca = PCA(n_components=K); X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 600) (60000,) (10000, 600) (10000,)


In [8]:
# Entrenamos un clasificador QDA con búsqueda aleatoria de hiperparámetros sobre los datos PCA de la Bda


from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)
# QDA: Clasificador prob. que asume clases gaussianas con covarianzas diferentes por clase
# Importa validación cruzada aleatoria y búsqueda de hiperparámetros
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time; start = time.time()

# BÚSQUEDA DE HIPERPARÁMETROS
clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)} # reg_param: parámetro de regularizacion. Evita matrices de covarianzas singulares
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23) # 
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32) # Prueba n_iter combinaciones aleatorias en paralelo (n_jobs = -1)

acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 76.58% con 0.0823
Tiempo (hh:mm:ss): 00:06:12


In [9]:
# Entrenamos un clasificador 

import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("zalando-datasets/fashion_mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 600
pca = PCA(n_components=K); X_train = pca.fit_transform(X_train) # primero calcula la matriz de covarianzas
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)


(60000, 600) (60000,) (10000, 600) (10000,)


In [10]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform; import time; start = time.time()
clf = LinearDiscriminantAnalysis(); G = {'tol': loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = RS.fit(X_train, y_train).score(X_test, y_test)
print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 81.59% con 0.0019
Tiempo (hh:mm:ss): 00:00:26


In [11]:







# Ejercicios: 
# CIFAR-10 (K = 600)

# Normalizamos las imágenes y apliamos PCA para reducir la dim de 784 a K componentes principales
import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("uoft-cs/cifar10").with_format("numpy")
X_train = ds['train'][:]['img'].astype(np.float32).reshape(-1, 32*32*3) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['img'].astype(np.float32).reshape(-1, 32*32*3) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 600; pca = PCA(n_components=K); X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(50000, 600) (50000,) (10000, 600) (10000,)


In [12]:
# Entrenamos un clasificador QDA con búsqueda aleatoria de hiperparámetros sobre los datos PCA de la Bda


from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)
# QDA: Clasificador prob. que asume clases gaussianas con covarianzas diferentes por clase
# Importa validación cruzada aleatoria y búsqueda de hiperparámetros
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time; start = time.time()

# BÚSQUEDA DE HIPERPARÁMETROS
clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)} # reg_param: parámetro de regularizacion. Evita matrices de covarianzas singulares
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23) # 
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32) # Prueba n_iter combinaciones aleatorias en paralelo (n_jobs = -1)

acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))


Precisión: 52.53% con 0.0008
Tiempo (hh:mm:ss): 00:05:13


In [13]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform; import time; start = time.time()
clf = LinearDiscriminantAnalysis(); G = {'tol': loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = RS.fit(X_train, y_train).score(X_test, y_test)
print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 40.91% con 0.0129
Tiempo (hh:mm:ss): 00:00:21
